In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd 

In [3]:
s1 = pd.read_csv("../student_resource/dataset/train/train_source1.tsv", sep='\t')
s2 = pd.read_csv("../student_resource/dataset/train/train_source2.tsv", sep='\t')
s3 = pd.read_csv("../student_resource/dataset/train/train_source3.tsv", sep='\t')
g = pd.read_csv("../student_resource/dataset/train/train_ground_truth.tsv", sep='\t')

## Normalization

In [4]:
import sys
sys.path.append("..")

from src.normalization import normalize_pipeline

In [5]:
s1_normalized = normalize_pipeline(s1)
s2_normalized = normalize_pipeline(s2)
s3_normalized = normalize_pipeline(s3)

In [ ]:
s1_normalized.head()

In [ ]:
s1_normalized['geo_block_key'].value_counts()

In [ ]:
s1_normalized['geo_block_key'].count()

## Standardization and Blocking

In [ ]:
from src.blocking import generate_global_candidate_pool, run_multi_source_geo_blocking

# 1. Global Scan (Shared tokenization & single-build indices)
# On SageMaker ml.r5.4xlarge: chunk_size=100000, n_threads=16
# Locally on M4: chunk_size=50000, n_threads=8
global_candidates_df = generate_global_candidate_pool(
    s1_normalized, 
    s2_normalized, 
    s3_normalized, 
    top_k=15, 
    chunk_size=100000, 
    n_threads=16
)

# 2. Local Spatial Safety Net
geo_candidates_df = run_multi_source_geo_blocking(
    s1_normalized, 
    s2_normalized, 
    s3_normalized, 
    top_k=15
)

# 3. Outer Merge into the Master Candidate Pool
print("\nMerging Local Geo matches into Master Candidate Pool...")
final_master_pool = pd.merge(
    global_candidates_df, 
    geo_candidates_df, 
    on=['entity_A', 'entity_B'], 
    how='outer'
).fillna(0.0)

print(f"Final Candidate Pool Size: {len(final_master_pool)}")

STAGES 1-3: GLOBAL BM25 CANDIDATE EXTRACTION

[INDIA | TEXT] S1: 883188 | S2: 2017799 | S3: 2115547
  -> Building Target S2 Index...


Split strings:   0%|          | 0/2017799 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/2017799 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/2017799 [00:00<?, ?it/s]

  -> Building Target S3 Index...


Split strings:   0%|          | 0/2115547 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/2115547 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/2115547 [00:00<?, ?it/s]

  -> Querying S1 across target indices in chunks of 100000...


Split strings:   0%|          | 0/100000 [00:00<?, ?it/s]

## Validation

In [ ]:
import pandas as pd

def validate_blocking_recall(candidates_df, ground_truth_df, s1_df, s2_df, s3_df):
    """
    Calculates fair recall by ensuring we only evaluate pairs where 
    both entities actually exist in the current sample dataframes.
    """
    print("Extracting available IDs from current sample subsets...")
    available_ids = set(s1_df['entity_id']).union(set(s2_df['entity_id'])).union(set(s3_df['entity_id']))
    
    print("Parsing raw ground truth clusters into verifiable pairs...")
    true_pairs = set()
    
    # Drop rows with NaN in matched_entity_ids
    valid_gt = ground_truth_df.dropna(subset=['matched_entity_ids'])
    
    for _, row in valid_gt.iterrows():
        s1_id = str(row['source1_entity_id']).strip()
        
        # Split the comma-separated string into a list of individual IDs
        matches = str(row['matched_entity_ids']).split(',')
        
        for match_id in matches:
            match_id = match_id.strip()
            if match_id:
                # ONLY count the pair if BOTH entities physically exist in our sample
                if s1_id in available_ids and match_id in available_ids:
                    pair = tuple(sorted([s1_id, match_id]))
                    true_pairs.add(pair)
                
    total_true = len(true_pairs)
    print(f"Total Verifiable True Matches to find: {total_true}")
    
    if total_true == 0:
        print("ERROR: No valid ground truth pairs exist in the current subsets.")
        return set(), set()

    print("Standardizing generated candidate pairs...")
    # Fast set comprehension for pairs
    candidate_pairs = set(
        tuple(sorted([str(a), str(b)])) 
        for a, b in zip(candidates_df['entity_A'], candidates_df['entity_B'])
    )
        
    total_candidates = len(candidate_pairs)
    print(f"Total Candidate Pairs Generated: {total_candidates}")
    
    # Calculate Recall
    captured_pairs = true_pairs.intersection(candidate_pairs)
    captured_count = len(captured_pairs)
    
    recall = (captured_count / total_true) * 100
    
    print("\n" + "="*30)
    print("BLOCKING PIPELINE METRICS")
    print("="*30)
    print(f"-> True Matches Captured: {captured_count} / {total_true}")
    print(f"-> RECALL: {recall:.2f}%")
    
    # Calculate Theoretical Search Space Reduction
    max_comps = (len(s1_df) * len(s2_df)) + (len(s1_df) * len(s3_df)) + (len(s2_df) * len(s3_df))
    reduction_ratio = 100 - ((total_candidates / max_comps) * 100) if max_comps > 0 else 0
    print(f"-> Search Space Reduction: ~{reduction_ratio:.4f}% (down from {max_comps:,} pairs)")
    
    return captured_pairs, true_pairs - candidate_pairs

# Execute validation on the deduplicated master pool
captured, missed = validate_blocking_recall(
    final_master_pool, 
    gt, 
    s1_normalized, 
    s2_normalized, 
    s3_normalized
)

## Generate baseline submission

In [ ]:
import pandas as pd

def generate_baseline_submission(master_pool_df: pd.DataFrame, s1_df: pd.DataFrame, output_path: str = "matching_results.tsv"):
    """
    Generates a leaderboard-ready TSV file from the candidate pool using naive thresholds.
    Ensures exact schema compliance: exactly one row per S1 entity, no duplicates.
    """
    print("Generating baseline submission for the leaderboard...")
    
    # 1. Naive Thresholding (Surrogate for LightGBM)
    # We now explicitly include the phonetic score threshold alongside text and geo scores.
    strong_matches = master_pool_df[
        (master_pool_df['bm25_text_score'] > 15.0) | 
        (master_pool_df['bm25_phonetic_score'] > 10.0) | 
        (master_pool_df.get('geo_ngram_score', pd.Series(0, index=master_pool_df.index)) > 0.5)
    ].copy()
    
    # 2. Group by Source 1 ID and create the comma-separated list of unique S2/S3 matches
    grouped = strong_matches.groupby('entity_A')['entity_B'].apply(
        lambda x: ','.join(x.dropna().unique())
    ).reset_index()
    
    grouped = grouped.rename(columns={
        'entity_A': 'source1_entity_id', 
        'entity_B': 'matched_entity_ids'
    })
    
    # 3. Ensure ALL Source 1 entities are present in the final file
    # This enforces the rule that every S1 entity must have exactly one row.
    submission_df = pd.DataFrame({'source1_entity_id': s1_df['entity_id']})
    submission_df = pd.merge(submission_df, grouped, on='source1_entity_id', how='left')
    
    # 4. Fill missing matches with an empty string
    # Entities with no matches are explicitly left blank.
    submission_df['matched_entity_ids'] = submission_df['matched_entity_ids'].fillna("")
    
    # 5. Export as TSV
    submission_df.to_csv(output_path, sep='\t', index=False)
    
    print(f"Saved submission to: {output_path}")
    print(f"Total rows in submission: {len(submission_df)} (Should match S1 row count: {len(s1_df)})")
    print(f"S1 entities with at least one match: {len(submission_df[submission_df['matched_entity_ids'] != ''])}")
    
    return submission_df

# Generate the final matching_results.tsv
submission = generate_baseline_submission(
    final_master_pool, 
    s1_normalized, 
    output_path="matching_results.tsv"
)